# 10 — Base analítica con proyecciones propias

**Objetivo:** construir una base entendible para corregir producción teórica con producción real reciente, clima y revisiones anteriores del pronóstico. Alcance principal: GAITANA exacta, producto/color/variedad, cinco horizontes semanales.

**Entradas:** `produccion_real_limpio.parquet`, `clima_limpio.parquet`, maestros comerciales locales y `proyecciones_semanales_modelo.parquet` del notebook de proyecciones de todos los años. Las limpiezas oficiales 05 y 02 conservan sus reglas ya contrastadas; no se vuelve a extraer ni a deduplicar arbitrariamente movimientos. No se leen planos ni curvas, ni se utiliza la antigua base de 108 columnas.

**Salidas fijas:** `modelo_con_proyecciones_teoricas/produccion_clima_semanal.parquet` (una fila por serie/semana observada) y `base_analitica_proyecciones.parquet` (una fila por serie/origen/horizonte, con targets faltantes conservados). Los notebooks 11 y 12 leen esas salidas; no dependen de este kernel.

Consultar la guía y el contexto del proyecto: faltante no es cero, GAITANA PPTO no es GAITANA, tallos reportados no equivalen automáticamente a exportable y un origen repetido no representa producción adicional.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'Datos_analiticos_proyecto').is_dir())
SALIDA=ROOT/'Datos_analiticos_proyecto'
DESTINO=SALIDA/'modelo_con_proyecciones_teoricas'
DESTINO.mkdir(exist_ok=True)
KEY=['finca','producto','color','variedad']
SEM=KEY+['semana_inicio']
pd.set_option('display.max_columns',30)


## 1. Producción real: cantidades y grano semanal

In [2]:
real = pd.read_parquet(SALIDA/"produccion_real_limpio.parquet",columns=KEY+["semana_inicio","fecha_corte","tallos","llave_completa"])
real = real.loc[real.finca.eq("GAITANA") & real.semana_inicio.ge("2021-01-01")].copy()
assert real.llave_completa.all()
assert real.tallos.notna().all(), 'Hay cantidades reales faltantes: revisar fuente antes de agregar.'
assert np.isfinite(real.tallos).all()
print('Movimientos con cantidades negativas (conservados para auditoría):',int(real.tallos.lt(0).sum()))
tallos_fuente = real.tallos.sum()
base = real.groupby(SEM,dropna=False).agg(
    tallos_reales=("tallos","sum"),registros_corte=("tallos","size"),
    dias_con_reporte=("fecha_corte","nunique")).reset_index()
assert not base.duplicated(SEM).any()
assert base.tallos_reales.sum()==tallos_fuente
referencia_real = base[SEM+["tallos_reales"]].copy()  # sólo memoria, nunca otra exportación
base["unidad"] = "tallos reportados"
base["produccion_observada"] = True
iso = base.semana_inicio.dt.isocalendar()
base["anio"],base["semana"] = iso.year,iso.week
print("Movimientos:",len(real),"Filas semanales:",len(base),"Tallos:",int(tallos_fuente))
display(base.groupby("producto").agg(filas=("tallos_reales","size"),tallos=("tallos_reales","sum"),
                                   variedades=("variedad","nunique")))
display(base.head())
del real
reconciliaciones = []


Movimientos con cantidades negativas (conservados para auditoría): 0


Movimientos: 4654552 Filas semanales: 44053 Tallos: 327991200


,filas,tallos,variedades
producto,,,
ACHILLEA,209,14149,4
AMAZON,256,653257,2
BARBATUS,1630,2193346,43
BRASSICA,7,982,2
CAMPANULA,194,89657,3
CARNATION,18849,60581236,278
FIORINO,8,2776,1
GREEN BALL,580,5839878,9
LEPIDIUM,56,36291,1


,finca,producto,color,variedad,semana_inicio,tallos_reales,registros_corte,dias_con_reporte,unidad,produccion_observada,anio,semana
0,GAITANA,ACHILLEA,LIGHT PINK,SASSY SUMMER TAFFY,2023-07-17,60,1,1,tallos reportados,True,2023,29
1,GAITANA,ACHILLEA,LIGHT PINK,SASSY SUMMER TAFFY,2023-07-24,185,4,3,tallos reportados,True,2023,30
2,GAITANA,ACHILLEA,LIGHT PINK,SASSY SUMMER TAFFY,2023-07-31,180,3,3,tallos reportados,True,2023,31
3,GAITANA,ACHILLEA,LIGHT PINK,SASSY SUMMER TAFFY,2023-08-07,120,2,2,tallos reportados,True,2023,32
4,GAITANA,ACHILLEA,LIGHT PINK,SASSY SUMMER TAFFY,2023-08-21,60,1,1,tallos reportados,True,2023,34


## 2. Clima: promedio diario y luego semanal

Se respetan mediciones válidas del limpio y se evita ponderar más los días con mayor frecuencia de muestreo. No se imputan observaciones climáticas futuras.

In [3]:
clima = pd.read_parquet(SALIDA/"clima_limpio.parquet")
clima = clima.loc[clima.finca.eq("GAITANA") & clima.medicion_valida].copy()
diario = clima.groupby(["finca","fecha","semana_inicio"]).agg(
    temperatura=("temperatura","mean"),humedad=("humedad","mean"),radiacion=("radiacion","mean"),
    temperatura_min=("temperatura_min","min"),temperatura_max=("temperatura_max","max"),
    mediciones=("instante","size"),hora_con_zona=("hora_con_zona","max")).reset_index()
clima_sem = diario.groupby(["finca","semana_inicio"]).agg(
    temperatura_semana=("temperatura","mean"),humedad_semana=("humedad","mean"),
    radiacion_semana=("radiacion","mean"),temperatura_min_semana=("temperatura_min","min"),
    temperatura_max_semana=("temperatura_max","max"),dias_clima=("fecha","nunique"),
    mediciones_clima=("mediciones","sum"),horario_clima_por_validar=("hora_con_zona","max")).reset_index()
antes = len(base)
base = base.merge(clima_sem,on=["finca","semana_inicio"],how="left",validate="many_to_one",indicator=True)
reconciliaciones.append({"union":"clima","cardinalidad":"N:1","antes":antes,"despues":len(base),
 "sin_pareja":int(base._merge.eq("left_only").sum()),"tallos":int(base.tallos_reales.sum())})
base = base.drop(columns="_merge")
assert len(base)==antes and base.tallos_reales.sum()==tallos_fuente
display(clima_sem.tail())
del clima, diario


,finca,semana_inicio,temperatura_semana,humedad_semana,radiacion_semana,temperatura_min_semana,temperatura_max_semana,dias_clima,mediciones_clima,horario_clima_por_validar
286,GAITANA,2026-06-29,14.381548,85.413690,155.187500,7.6,21.9,7,336,False
287,GAITANA,2026-07-06,14.685714,83.169643,154.958333,5.1,21.2,7,336,False
288,GAITANA,2026-07-13,14.810714,80.752976,165.738095,6.1,21.7,7,336,False
289,GAITANA,2026-07-20,14.289583,84.089286,167.639881,3.9,21.6,7,336,False
290,GAITANA,2026-07-27,11.737595,86.712121,165.903409,5.4,20.8,2,70,False


## 3. Calendario y variables conocidas al origen

H1 es la semana que comienza en el origen; H5 es cuatro semanas después. Los rezagos son de calendario, no la fila anterior disponible. Se conservan las semanas sin reporte como faltantes. Temperatura, humedad y radiación requieren cinco días de reporte; sólo entran lag1 y media de cuatro semanas cerradas. No se usan plantas, edades ni potenciales antiguos.

In [4]:
def construir_casos(base, retraso_semanas=0):
    """Una fila = serie/origen/horizonte. Ninguna X usa la producción objetivo."""
    key = ["finca","producto","color","variedad"]
    b = base.copy()
    b["semana_inicio"] = b.semana_inicio.astype("datetime64[ns]")
    calendario = pd.DataFrame({"origen":pd.date_range(b.semana_inicio.min(),
                        b.semana_inicio.max()+pd.Timedelta(weeks=1),freq="W-MON")})
    calendario["origen"] = calendario.origen.astype("datetime64[ns]")
    grid = b[key].drop_duplicates().merge(calendario,how="cross")
    grid = grid.merge(b[key+["semana_inicio","tallos_reales"]],
                      left_on=key+["origen"],right_on=key+["semana_inicio"],
                      how="left",validate="one_to_one").sort_values(key+["origen"])
    g = grid.groupby(key,sort=False).tallos_reales
    desfase = 1+retraso_semanas
    for lag in [1,2,3]:
        grid[f"produccion_lag{lag}"] = g.shift(lag+retraso_semanas)
    for v in [4,8]:
        grid[f"media_{v}"] = g.transform(lambda s:s.shift(desfase).rolling(v,min_periods=v).mean())
    grid["desv_4"] = g.transform(lambda s:s.shift(desfase).rolling(4,min_periods=4).std())
    grid["tendencia_4_8"] = grid.media_4-grid.media_8
    grid["cambio_reciente"] = grid.produccion_lag1-grid.produccion_lag2
    grid["observaciones_previas"] = g.transform(lambda s:s.notna().cumsum().shift(desfase,fill_value=0))
    grid = grid.drop(columns=["semana_inicio","tallos_reales"])

    # Cada finca/semana aparece en muchas variedades, pero clima sólo se cuenta una vez.
    cols = ["temperatura_semana","humedad_semana","radiacion_semana"]
    assert b.groupby(["finca","semana_inicio"])[cols].nunique().le(1).all().all()
    clima = b.groupby(["finca","semana_inicio"])[cols+["dias_clima"]].first().reset_index()
    clima.loc[clima.dias_clima.lt(5),cols] = np.nan  # una semana muy incompleta no representa su promedio
    clima = b[["finca"]].drop_duplicates().merge(calendario,how="cross").merge(
        clima.rename(columns={"semana_inicio":"origen"}),on=["finca","origen"],how="left",validate="one_to_one").sort_values(["finca","origen"])
    for col in cols:
        clima[col+"_lag1"] = clima.groupby("finca")[col].shift(desfase)
        clima[col+"_media4"] = clima.groupby("finca")[col].transform(lambda s:s.shift(desfase).rolling(4,min_periods=4).mean())
    clima = clima.drop(columns=cols+["dias_clima"])
    grid = grid.merge(clima,on=["finca","origen"],how="left",validate="many_to_one")

    etiquetas = b[key+["semana_inicio","tallos_reales"]].rename(columns={"semana_inicio":"semana_objetivo","tallos_reales":"y"})
    casos = []
    for h in range(1,6):
        d = grid.loc[grid.observaciones_previas.gt(0)].copy()
        d["horizonte"] = h
        d["semana_objetivo"] = d.origen+pd.Timedelta(weeks=h-1)
        d["ultima_semana_utilizada"] = d.origen-pd.Timedelta(weeks=desfase)
        d = d.merge(etiquetas,on=key+["semana_objetivo"],how="left",validate="many_to_one")
        d["fecha_estacional"] = d.semana_objetivo-pd.Timedelta(weeks=52)
        estacional = etiquetas.rename(columns={"semana_objetivo":"fecha_estacional","y":"produccion_52"})
        d = d.merge(estacional,on=key+["fecha_estacional"],how="left",validate="many_to_one")
        assert (d.fecha_estacional+pd.Timedelta(weeks=1+retraso_semanas)).le(d.origen).all()
        semana = d.semana_objetivo.dt.isocalendar().week.astype(float)
        d["semana_seno"] = np.sin(2*np.pi*semana/52)
        d["semana_coseno"] = np.cos(2*np.pi*semana/52)
        casos.append(d)
    casos = pd.concat(casos,ignore_index=True)
    assert not casos.duplicated(key+["origen","horizonte"]).any()
    assert casos.ultima_semana_utilizada.lt(casos.origen).all()
    return casos

categoricas = ["producto","color","variedad"]
temporales = ["horizonte","semana_seno","semana_coseno"]
memoria = ["produccion_lag1","produccion_lag2","produccion_lag3","media_4","media_8",
           "desv_4","tendencia_4_8","cambio_reciente","produccion_52"]
agronomicas = []
climaticas = [v+s for v in ["temperatura_semana","humedad_semana","radiacion_semana"] for s in ["_lag1","_media4"]]
X_columnas = categoricas+temporales+memoria+agronomicas+climaticas
assert "y" not in X_columnas

casos = construir_casos(base)
for col in temporales+memoria+agronomicas+climaticas:
    casos[col] = casos[col].astype(np.float32)
admisible = casos.y.notna() & casos.media_4.notna() & casos.observaciones_previas.ge(26)
datos_modelo = casos.loc[admisible].copy()
print("Casos con target:",int(casos.y.notna().sum()),"Admisibles:",len(datos_modelo))
display(datos_modelo[X_columnas].isna().mean().mul(100).round(1).to_frame("faltantes_pct"))


Casos con target: 211001 Admisibles: 143897


,faltantes_pct
producto,0.0
color,0.0
variedad,0.0
horizonte,0.0
semana_seno,0.0
semana_coseno,0.0
produccion_lag1,0.0
produccion_lag2,0.0
produccion_lag3,0.0
media_4,0.0


## 4. Homologación de nombres comerciales

Se mantienen las claves de producción. Los maestros y alias explícitos construyen claves auxiliares sin coincidencias aproximadas ni elección arbitraria entre variedades ambiguas.

El color productivo se mantiene incluso si difiere del maestro: se aplican sólo alias explícitos. No se asigna una misma proyección a dos series productivas de colores diferentes. Identidades que sigan siendo ambiguas se excluyen de la unión y se muestran, sin eliminar su producción.

In [5]:
import numpy as np
import re
from datetime import date
import matplotlib.pyplot as plt

RUTA_MAESTROS = ROOT / 'Notebooks' / 'Extracción SQL' / 'Datos' / '01_maestros_y_geografia'
import json
CATALOGO=RUTA_MAESTROS/'Catalogo_analitico'
assert (CATALOGO/'maestro_variedades_integrado.parquet').exists(), 'Ejecutar primero 00_catalogo_variedades.ipynb'
maestro_productos=pd.read_parquet(CATALOGO/'productos.parquet')
maestro_colores=pd.read_parquet(CATALOGO/'colores.parquet')
maestro=pd.read_parquet(CATALOGO/'maestro_variedades_integrado.parquet')

def normalizar(s):
    return (s.astype('string').str.strip().str.upper().str.normalize('NFKD')
            .str.encode('ascii', errors='ignore').str.decode('ascii')
            .str.replace(r'\s+', ' ', regex=True).replace('', pd.NA))

def vacios(d):
    return d.replace(['','NULL','null','None','NONE','N/A','n/a','NA','na'], np.nan).replace(r'^\s*$', np.nan, regex=True)

def campo(d, *nombres):
    cols = dict(zip(normalizar(pd.Series(d.columns)), d.columns))
    for nombre in normalizar(pd.Series(nombres)):
        if nombre in cols:
            return d[cols[nombre]]
    return pd.Series(pd.NA, index=d.index, dtype='object')

def fechas(s):
    # No interpretar seriales Excel como nanosegundos desde 1970.
    numero = pd.to_numeric(s, errors='coerce')
    f = pd.to_datetime(s.where(numero.isna()), errors='coerce', format='mixed', dayfirst=True)
    serial = numero.between(20000, 80000)
    f.loc[serial] = pd.to_datetime(numero.loc[serial], unit='D', origin='1899-12-30')
    return f.where(f.between('1980-01-01','2100-12-31'))

def lunes_iso(anio, semana):
    # Una semana imposible (p. ej. 53 en ciertos años) se reporta, no se desplaza silenciosamente.
    pares = pd.DataFrame({'a':pd.to_numeric(anio, errors='coerce'), 's':pd.to_numeric(semana, errors='coerce')}).fillna(-1)
    def convertir(t):
        a,s=t
        try:
            if pd.isna(a) or pd.isna(s) or a != int(a) or s != int(s) or not 1980 <= a <= 2100: return pd.NaT
            return pd.Timestamp(date.fromisocalendar(int(a), int(s), 1))
        except ValueError:
            return pd.NaT
    mapa = {t:convertir(t) for t in pares.itertuples(index=False, name=None)}
    return pd.Series([mapa[t] for t in pares.itertuples(index=False, name=None)], index=anio.index, dtype='datetime64[ns]')

equivalencias=json.loads((CATALOGO/'equivalencias.json').read_text(encoding='utf-8'))
ALIAS_PRODUCTOS=equivalencias['productos']
ALIAS_COLORES=equivalencias['colores']
# Rosado y lila no se traducen por intuición: la variedad resuelta aporta su color oficial.
maestro['_var'] = normalizar(maestro.NomVariedad)
maestro['_prod'] = normalizar(maestro.NomProducto)
maestro['_color'] = normalizar(maestro.NomColor)
por_variedad = {k:g for k,g in maestro.groupby('_var')}
def mapa_unico(d, nombre, identificador):
    t=d.assign(k=normalizar(d[nombre]))
    t=t.loc[~t.k.duplicated(keep=False) & t.k.notna()]
    return t.set_index('k')[identificador].to_dict()
ids_producto = mapa_unico(maestro_productos,'NomProducto','IdProducto')
ids_color = mapa_unico(maestro_colores,'NomColor','IdColor')
nom_producto = dict(zip(normalizar(maestro_productos.NomProducto),maestro_productos.NomProducto))
nom_color = dict(zip(normalizar(maestro_colores.NomColor),maestro_colores.NomColor))

def homologar(d):
    d=d.copy()
    for c in ['producto','color','variedad']:d[c+'_original']=d[c]
    claves=pd.DataFrame({'_prod':normalizar(d.producto).replace(ALIAS_PRODUCTOS),
        '_color':normalizar(d.color).replace(ALIAS_COLORES),'_var':normalizar(d.variedad)})
    res=[]
    for prod,col,var in claves.fillna('').drop_duplicates().itertuples(index=False,name=None):
        candidatas=por_variedad.get(var,maestro.iloc[:0])
        if prod:candidatas=candidatas.loc[candidatas._prod.eq(prod)]
        inicial=len(candidatas)
        comerciales=candidatas[['_prod','_color','_var']].drop_duplicates()
        # Diferentes IDs no implican distintas identidades comerciales.
        if len(comerciales)>1 and col:
            candidatas=candidatas.loc[candidatas._color.eq(col)]
            comerciales=candidatas[['_prod','_color','_var']].drop_duplicates()
        r={'_prod':prod,'_color':col,'_var':var,
           'estado_maestro':'sin_coincidencia' if inicial==0 else 'ambiguo',
           'IdVariedad':np.nan,'IdProducto':ids_producto.get(prod,np.nan),'IdColor':ids_color.get(col,np.nan),
           'producto':nom_producto.get(prod,prod or pd.NA),'color':nom_color.get(col,col or pd.NA),
           'variedad':var or pd.NA,'conflicto_color':False,'clave_curva':pd.NA,
           'ids_maestro_equivalentes':'','identidad_comercial_sin_id_unico':False}
        if len(comerciales)==1:
            identidad=comerciales.iloc[0]
            ids=sorted(candidatas.IdVariedad.unique())
            unico=len(ids)==1
            # No escoger arbitrariamente un código SQL cuando existen varios.
            r.update(IdVariedad=ids[0] if unico else np.nan,
                IdProducto=candidatas.IdProducto.iloc[0] if candidatas.IdProducto.nunique()==1 else np.nan,
                IdColor=candidatas.IdColor.iloc[0] if candidatas.IdColor.nunique()==1 else np.nan,
                producto=nom_producto.get(identidad._prod,identidad._prod),
                color=nom_color.get(identidad._color,identidad._color),
                variedad=candidatas.NomVariedad.iloc[0] if unico else identidad._var,
                estado_maestro='unico' if unico else 'comercial_unica_ids_multiples',
                conflicto_color=bool(col and col!=identidad._color),
                clave_curva='COMERCIAL|'+identidad._prod+'|'+identidad._color+'|'+identidad._var,
                ids_maestro_equivalentes=';'.join(str(int(x)) for x in ids),
                identidad_comercial_sin_id_unico=not unico)
        res.append(r)
    m=claves.fillna('').merge(pd.DataFrame(res),on=['_prod','_color','_var'],how='left',validate='many_to_one')
    for c in ['IdVariedad','IdProducto','IdColor','producto','color','variedad','estado_maestro','conflicto_color','clave_curva','ids_maestro_equivalentes','identidad_comercial_sin_id_unico']:
        d[c]=m[c].to_numpy()
    return d


## 5. Incorporar proyecciones y su historia

La fecha de disponibilidad viene incluida en el insumo de proyecciones: máximo entre cierre nominal y día posterior a AL1, bajo el supuesto documentado del proyecto. No acredita publicación real. Se elige el último snapshot disponible, con tolerancia de cuatro semanas, y después se busca la semana objetivo exacta. No se recupera una versión antigua sólo porque tenga mayor cobertura.

Una curva es completa si todas las cohortes del grupo son proyectables o corresponden a siembras posteriores con cero esperado. El valor parcial se conserva para diagnóstico y no se usa como baseline completo. Las proyecciones anteriores corresponden a la misma semana objetivo y sólo a snapshots disponibles. Sus diferencias miden revisiones del pronóstico, no errores observados.

Las ediciones anuales de curvas usan cortes anteriores al plano, pero su publicación histórica no está certificada. Esta limitación acompaña toda evaluación.

In [6]:
PROY=ROOT/'Notebooks'/'Proyecciones Teoricas Propias'/'resultados_todos_los_anios'
teoricas=pd.read_parquet(PROY/'proyecciones_semanales_modelo.parquet')
mapa=homologar(base[KEY].drop_duplicates())
aux=['union_'+c for c in KEY]
for c,k in zip(KEY,aux):
    # El maestro no debe convertir dos colores productivos en la misma serie.
    fuente=mapa.color_original if c=='color' else mapa[c]
    mapa[k]=normalizar(fuente).replace(ALIAS_COLORES) if c=='color' else normalizar(fuente)
ambiguas=mapa.duplicated(aux,keep=False)
display(mapa.loc[ambiguas,KEY+[c+'_original' for c in ['producto','color','variedad']]])
print('Series excluidas de unión por identidad ambigua:',int(ambiguas.sum()))
mapa=mapa.loc[~ambiguas].copy()
# Restaurar las etiquetas originales que utiliza el modelo 09.
for c in ['producto','color','variedad']: mapa[c]=mapa[c+'_original']
mapa=mapa[KEY+aux]
assert not mapa.duplicated(KEY).any()
for c,k in zip(KEY,aux): teoricas[k]=normalizar(teoricas[c])
snapshots=teoricas[['plano','fecha_plano','disponible']].drop_duplicates()
assert not snapshots.duplicated('plano').any()
teoricas=teoricas.drop(columns='disponible')
display(snapshots.assign(con_AL1=snapshots.disponible.notna()).groupby(snapshots.fecha_plano.dt.year).con_AL1.agg(['size','sum']))
disponibles=snapshots.dropna(subset=['disponible']).sort_values(['disponible','fecha_plano']).drop_duplicates('disponible',keep='last')
for c in ['disponible','fecha_plano']: disponibles[c]=disponibles[c].astype('datetime64[ns]')
origenes=pd.DataFrame({'origen':sorted(casos.origen.unique())})
eleccion=pd.merge_asof(origenes.sort_values('origen'),disponibles.sort_values('disponible'),left_on='origen',right_on='disponible',direction='backward',tolerance=pd.Timedelta(weeks=4))
assert (eleccion.disponible.isna() | eleccion.disponible.le(eleccion.origen)).all()
# Reagrupar después de normalizar evita multiplicar filas por diferencias tipográficas.
teoricas=teoricas.groupby(['plano','semana_proyeccion']+aux,dropna=False,as_index=False).agg(teoria=('teoria','sum'),valores=('teoria','count'),sin_cobertura=('sin_cobertura','sum'),cohortes=('cohortes','sum'))
teoricas['teoria']=teoricas.teoria.where(teoricas.valores.gt(0))
teoricas=teoricas.drop(columns='valores')
# Historia de pronósticos de la MISMA semana objetivo, ordenada por disponibilidad.
teoricas=teoricas.merge(snapshots[['plano','disponible']],on='plano',how='left',validate='many_to_one')
teoricas=teoricas.dropna(subset=['disponible']).sort_values(['disponible','plano'])
teoricas['valor_completo']=teoricas.teoria.where(teoricas.sin_cobertura.eq(0))
g=teoricas.groupby(aux+['semana_proyeccion'],dropna=False)
for lag in [1,2]:
    teoricas[f'teoria_previa_{lag}']=g.valor_completo.shift(lag)
    fecha_previa=g.disponible.shift(lag)
    assert (fecha_previa.isna()|fecha_previa.le(teoricas.disponible)).all()
teoricas['revision_teoria']=teoricas.valor_completo-teoricas.teoria_previa_1
teoricas=teoricas.drop(columns=['disponible','valor_completo'])
integrados=casos.merge(mapa,on=KEY,how='left',validate='many_to_one').merge(eleccion,on='origen',how='left',validate='many_to_one')
integrados=integrados.merge(teoricas.rename(columns={'semana_proyeccion':'semana_objetivo'}),on=['plano','semana_objetivo']+aux,how='left',validate='many_to_one')
assert len(integrados)==len(casos)
integrados['curva_completa']=integrados.teoria.notna() & integrados.sin_cobertura.eq(0)
integrados['antiguedad_curva']=(integrados.origen-integrados.fecha_plano).dt.days/7
integrados['teoria_modelo']=integrados.teoria.where(integrados.curva_completa)
integrados['desviacion_reciente_teoria']=integrados.media_4-integrados.teoria_modelo
admisible=integrados.y.notna()&integrados.media_4.notna()&integrados.observaciones_previas.ge(26)
datos=integrados.loc[admisible].copy()
X_teoria=X_columnas+['teoria_modelo','antiguedad_curva','desviacion_reciente_teoria']
X_historia=X_teoria+['teoria_previa_1','teoria_previa_2','revision_teoria']
display(datos.groupby(datos.origen.dt.year).agg(casos=('y','size'),con_curva_completa=('curva_completa','sum')))


,finca,producto,color,variedad,producto_original,color_original,variedad_original


Series excluidas de unión por identidad ambigua: 0


,size,sum
fecha_plano,,
2021,52,52
2022,51,50
2023,52,52
2024,53,53
2025,52,52
2026,32,32


,casos,con_curva_completa
origen,,
2021,11589,3838
2022,22375,5686
2023,29777,9803
2024,30204,13209
2025,30778,7536
2026,19174,1643


## 6. Auditoría, prueba temporal y exportación

La tabla semanal conserva cada cantidad real. La tabla de casos repite objetivos desde distintos orígenes: sus tallos nunca se suman como inventario físico. La prueba temporal modifica producción y clima futuros y verifica que las variables anteriores no cambian.

In [7]:
assert not base.duplicated(SEM).any()
pd.testing.assert_frame_equal(referencia_real.sort_values(SEM).reset_index(drop=True),base[SEM+['tallos_reales']].sort_values(SEM).reset_index(drop=True))
llave_caso=KEY+['origen','semana_objetivo','horizonte']
assert not integrados.duplicated(llave_caso).any()
assert integrados.ultima_semana_utilizada.lt(integrados.origen).all()
assert (integrados.disponible.isna()|integrados.disponible.le(integrados.origen)).all()
corte_prueba=pd.Timestamp('2026-01-05')
# Prueba acotada a tres series para verificar el calendario y los rezagos.
claves_prueba=base[KEY].drop_duplicates().head(3)
muestra=base.merge(claves_prueba,on=KEY,validate='many_to_one')
alterada=muestra.copy()
futuras=alterada.semana_inicio.ge(corte_prueba)
alterada.loc[futuras,'tallos_reales']=alterada.loc[futuras,'tallos_reales']+123456
for c in ['temperatura_semana','humedad_semana','radiacion_semana']: alterada.loc[futuras,c]=99999
a=construir_casos(muestra);z=construir_casos(alterada)
cols=llave_caso+X_columnas
cols=list(dict.fromkeys(cols))
pd.testing.assert_frame_equal(a.loc[a.origen.le(corte_prueba),cols].reset_index(drop=True),z.loc[z.origen.le(corte_prueba),cols].reset_index(drop=True))
integrados['admisible_modelo']=integrados.media_4.notna()&integrados.observaciones_previas.ge(26)
integrados['motivo_cobertura']=np.select([integrados.plano.isna(),integrados.cohortes.isna(),~integrados.curva_completa],['sin_snapshot_disponible','sin_pareja_comercial_u_objetivo','cohortes_sin_curva'],default='curva_completa')
display(integrados.groupby('motivo_cobertura').agg(casos=('origen','size'),con_real=('y','count')))
display(integrados.loc[integrados.y.notna()].groupby('producto').agg(casos=('y','size'),con_curva=('curva_completa','sum'),admisibles=('admisible_modelo','sum')))
base.to_parquet(DESTINO/'produccion_clima_semanal.parquet',index=False)
integrados.to_parquet(DESTINO/'base_analitica_proyecciones.parquet',index=False)
print('Conservación por llave y prueba temporal: correctas.')
print('Panel semanal:',base.shape,'; casos de pronóstico:',integrados.shape)
print('Último reporte:',base.semana_inicio.max(),'; último origen:',integrados.origen.max())


,casos,con_real
motivo_cobertura,,
cohortes_sin_curva,229938,159656
curva_completa,47677,44848
sin_pareja_comercial_u_objetivo,342565,5523
sin_snapshot_disponible,1015,974


,casos,con_curva,admisibles
producto,,,
ACHILLEA,989,0,475
AMAZON,1260,179,1115
BARBATUS,7538,918,5229
BRASSICA,10,0,0
CAMPANULA,925,0,550
CARNATION,90082,19891,59572
FIORINO,35,0,0
GREEN BALL,2774,995,1945
LEPIDIUM,268,0,60


Conservación por llave y prueba temporal: correctas.
Panel semanal: (44053, 20) ; casos de pronóstico: (621195, 47)
Último reporte: 2026-08-31 00:00:00 ; último origen: 2026-09-07 00:00:00
